In [ ]:
# ==============================================================================
# 🌟 AI 코딩 튜터가 준비한 초급 실습! 🌟
#
# [데이터셋 소개]
# 데이터셋명: lemon-mint/korean_english_parallel_wiki_augmented_v1
# 의미: 한국어-영어 병렬 위키 데이터셋 (Korean-English Parallel Wiki Augmented)
# 설명: 이 데이터셋은 위키피디아에서 추출한 대규모의 한국어-영어 병렬 문장 쌍으로 구성되어 있어요.
#       'translation' 태스크에 최적화되어 있으며, 기계 번역 모델(NMT)을 훈련하거나
#       두 언어 간의 의미적 유사성을 분석할 때 아주 유용합니다!
#
# [🎉 오늘 실습 목표]
# 1. 데이터셋을 빠르고 효율적으로 불러오기 (스트리밍 학습).
# 2. 데이터의 통계적 특성을 분석해보기.
# 3. 주어진 문장 쌍을 가지고, 마치 LLM에게 프롬프트를 짜주듯이
#    '이 번역을 평가해줘!'라는 코딩 능력을 보여주는 창의적인 실습을 진행합니다.
#
# ✨ 이 코드를 실행하면 데이터의 '가장 흥미로운 부분'을 발견할 수 있을 거예요! ✨
# ==============================================================================

import random
import numpy as np
from datasets import load_dataset, get_dataset_config_names
from typing import List, Dict, Any

# --- ⚙️ 환경 설정 및 상수 정의 ---
DATASET_NAME = "lemon-mint/korean_english_parallel_wiki_augmented_v1"
SAMPLE_COUNT = 100 # 전체 데이터셋이 크기 때문에, 재미있게 관찰할 샘플 100개만 가져와요!

print("=" * 80)
print(f"📚 데이터셋 로드 준비: {DATASET_NAME}")
print("=" * 80)

# 1. 데이터셋 이름 설정 및 Config 확인 (필수 단계)
# 데이터셋 ID를 사용합니다.
DATASET_ID = DATASET_NAME

try:
    # Config 목록을 가져옵니다.
    configs = get_dataset_config_names(DATASET_ID)
    print(f"✅ 사용 가능한 Config 목록: {configs}")
    selected_config = configs[0]
except Exception as e:
    print(f"ℹ️ Config 확인 중 오류 발생 또는 기본 설정만 제공됩니다. 기본 설정을 사용합니다.")
    selected_config = None

# 2. 데이터 로드 (스트리밍 vs. 샘플 로딩 전략)
print("\n--- 🚀 Step 1: 데이터셋 로딩 전략 (스트리밍 시도) ---")
dataset = None
try:
    # 🌟 최적의 로딩 방법: streaming=True를 사용하여 메모리 효율을 극대화합니다.
    print("Attempting to load dataset using streaming=True (efficient mode)...")
    # split='train'을 지정하고 스트리밍을 시도합니다.
    dataset = load_dataset(DATASET_NAME, split='train', streaming=True)
    print("✨ 성공! 데이터셋을 스트리밍 모드(Streaming)로 로드했습니다. 메모리 걱정 없이 대용량 데이터 처리가 가능해요!")

except Exception as e:
    # 스트리밍 로드가 실패하거나 복잡할 경우, 안전하게 소량만 다운로드하여 진행합니다.
    print(f"⚠️ 경고: 스트리밍 로드 실패 ({type(e).__name__}). 안전하게 소량 데이터셋으로 폴백(Fallback) 합니다. ({e})")
    try:
        # streaming=False로 설정하고, 테스트 환경 등에서 작은 샘플을 강제 다운로드합니다.
        dataset = load_dataset(DATASET_NAME, split='train')
        print("✨ 성공! 일반 데이터셋 모드로 소량의 데이터를 다운로드하여 진행합니다.")
    except Exception as fallback_e:
        print(f"🛑 치명적인 오류: 데이터셋을 로드할 수 없습니다. {fallback_e}")
        exit()

# 3. 데이터 샘플링 및 준비
print("\n--- 🔬 Step 2: 데이터 샘플링 및 초기 분석 ---")
# ❌ 절대로 len(dataset)를 사용하지 마세요! (스트리밍에서는 불가능해요!)
# ⭕ 대신 dataset.take(K)를 사용해서 상위 K개 샘플만 가져오는 것이 가장 안전하고 빠릅니다.
sampled_dataset_iterator = dataset.take(SAMPLE_COUNT)
sample_data_list = list(sampled_dataset_iterator)

if not sample_data_list:
    print("❌ 샘플 데이터를 가져오는 데 실패했습니다. 실행을 종료합니다.")
    exit()

# 📊 정량적 분석 1: 샘플 개수 확인
print(f"✅ 총 분석 샘플 개수: {len(sample_data_list)}개 (처음 {SAMPLE_COUNT}개만 가져왔어요!)")

# 📊 정량적 분석 2: 스코어(score) 컬럼의 통계 분석
scores = [sample['score'] for sample in sample_data_list]
mean_score = np.mean(scores)
std_score = np.std(scores)
print(f"🔢 통계 분석: 'score' 컬럼의 평균 점수: {mean_score:.4f}, 표준편차: {std_score:.4f}")
print("   (score는 번역 모델의 품질 등을 나타내는 점수로 추정됩니다.)")


# ==============================================================================
# 🧠 창의적 실습: AI '번역 품질 평가 봇' 만들기 (Prompt Engineering & 분석)
# ==============================================================================
print("\n" + "=" * 80)
print("🧠 Step 3: AI '번역 품질 평가 봇' 만들기 실습")
print("=" * 80)
print("👉 목표: 하나의 문장 쌍을 가져와서, 단순히 데이터만 보여주는 것이 아니라")
print("   '이걸 가지고 AI한테 질문을 던지면 어떨까?'라는 관점에서 분석하는 겁니다!")

def create_evaluation_prompt(korean_text: str, english_text: str, score: float) -> str:
    """
    주어진 병렬 문장 쌍을 분석하여, LLM(대규모 언어 모델)에게 번역 평가를 요청하는 프롬프트를 생성합니다.
    
    Args:
        korean_text: 한국어 원문.
        english_text: 영어 원문.
        score: 데이터셋이 부여한 점수.
    """
    # ✨ 위트 있는 주석: LLM 프롬프트는 질문 형식으로 작성해야 효과적입니다!
    prompt = f"""
    [번역 품질 평가 요청]
    다음은 영어와 한국어의 병렬 문장 쌍입니다.
    영어 원문 (Source): '{english_text}'
    한국어 번역 (Target): '{korean_text}'
    
    1. 자연스러움 평가: 한국어 번역이 한국어 화자 입장에서 어색한 부분이 있는지 평가해주세요.
    2. 정확도 평가: 원문(Source)의 핵심 의미가 번역(Target)에 빠짐없이 담겨 있는지 점검해주세요.
    3. 추천 개선안: 만약 개선할 부분이 있다면, 가장 자연스럽고 정확한 한국어 문장으로 수정하여 제안해주세요.
    
    [메타 정보]
    제공된 'score' 값은 {score:.2f}점입니다. 이 점수를 고려하여 총평을 내려주세요.
    """
    return prompt

print(f"\n[👀 첫 번째 샘플 테스트]: {sample_data_list[0]['english'][:40]}...")
print("--------------------------------------------------------------------------------")

# 반복문을 통해 실습 진행
for i, sample in enumerate(sample_data_list):
    if i > 5: # 너무 많은 출력을 막기 위해 처음 6개만 보여줍니다.
        break
        
    english = sample['english']
    korean = sample['korean']
    score = sample['score']
    
    # 💡 창의적인 활용: 번역 품질 평가 프롬프트 생성
    evaluation_prompt = create_evaluation_prompt(korean, english, score)
    
    print(f"\n[✨ 샘플 {i+1} 분석 (Score: {score:.2f})]")
    print(f"  [English]: {english}")
    print(f"  [Korean]: {korean}")
    
    print("\n>>> 🤖 LLM 평가 요청 프롬프트 (Prompt for LLM) <<<")
    # 코드를 실행하는 것 자체가 프롬프트 설계 훈련이 됩니다!
    print(evaluation_prompt)
    print("--------------------------------------------------------------------------------")

print("\n🎉 🎉 실습 완료! 🎉 🎉")
print("축하합니다! 당신은 단순한 데이터 로딩을 넘어, 데이터의 '가치'를 끌어내는 분석가가 되었습니다.")
print("지금까지의 작업은 데이터를 단순히 저장된 텍스트로 본 것이 아니라,")
print("AI 모델에게 질문을 던질 수 있는 '지식의 원천'으로 활용하는 훈련이었습니다.")
print("다음 단계에서는 이 프롬프트를 실제로 OpenAI나 HuggingFace의 모델 API에 연결해보면 완벽할 거예요!")